# Full RAG + Uncertainty Estimation Pipeline  
### Using Qwen2.5-7B-Instruct (bf16), SAFE-Lite, MARS (TruthTorch or fallback), ECC  
### IR Project — Arnes HPC Jupyter Environment


In [3]:
import os

# HuggingFace cache redirection (MUST be FIRST cell)
os.environ["HF_HOME"] = "/d/hpc/projects/FRI/ma76193/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_transformers"

# Create folders if missing
for path in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    os.makedirs(path, exist_ok=True)

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE =", os.environ["TRANSFORMERS_CACHE"])


HF_HOME = /d/hpc/projects/FRI/ma76193/hf_home
HF_DATASETS_CACHE = /d/hpc/projects/FRI/ma76193/hf_datasets
TRANSFORMERS_CACHE = /d/hpc/projects/FRI/ma76193/hf_transformers


In [4]:
import os
import sys
import json
import time
import random
import logging
import threading
import subprocess
from pathlib import Path
from datetime import datetime

JAVA = "/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12"

os.environ["JAVA_HOME"] = JAVA
os.environ["JVM_PATH"] = f"{JAVA}/lib/server/libjvm.so"
os.environ["LD_LIBRARY_PATH"] = f"{JAVA}/lib/server:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PATH"] = f"{JAVA}/bin:" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
!java -version
!find $JAVA_HOME -name "libjvm.so"

import torch

# Logging
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "full_notebook.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
              logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("nb")

def banner(msg):
    line = "=" * 80
    log.info(line)
    log.info(f"*** {msg} ***")
    log.info(line)


JAVA_HOME = /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12
openjdk version "21.0.1" 2023-10-17 LTS
OpenJDK Runtime Environment Temurin-21.0.1+12 (build 21.0.1+12-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.1+12 (build 21.0.1+12-LTS, mixed mode, sharing)
/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so


In [5]:
!wget -O /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/openjdk21.tar.gz \
    https://github.com/adoptium/temurin21-binaries/releases/download/jdk-21.0.1%2B12/OpenJDK21U-jdk_x64_linux_hotspot_21.0.1_12.tar.gz
!tar -xzf /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/openjdk21.tar.gz \
    -C /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/
!ls /d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311


--2025-12-10 15:08:25--  https://github.com/adoptium/temurin21-binaries/releases/download/jdk-21.0.1%2B12/OpenJDK21U-jdk_x64_linux_hotspot_21.0.1_12.tar.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/602574963/3a83d4fc-3866-4f26-9e68-e6bfc7ba7993?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-10T14%3A51%3A09Z&rscd=attachment%3B+filename%3DOpenJDK21U-jdk_x64_linux_hotspot_21.0.1_12.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-10T13%3A50%3A23Z&ske=2025-12-10T14%3A51%3A09Z&sks=b&skv=2018-11-09&sig=7KYBwMjD%2B9itD24%2B%2BhmpYXu60MJ1rwVcK2f4C9eiehs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSI

In [6]:
# Heartbeat to show notebook is alive during long retrieval/index steps
_stop_hb = threading.Event()

def _heartbeat(period=30):
    while not _stop_hb.is_set():
        log.info("[HEARTBEAT] Notebook run alive...")
        time.sleep(period)

hb = threading.Thread(target=_heartbeat, daemon=True)
hb.start()



2025-12-10 15:08:35,912 | INFO | [HEARTBEAT] Notebook run alive...


In [7]:
!pip install ipywidgets


Defaulting to user installation because normal site-packages is not writeable


In [8]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Xms1g -Xmx8g")

log.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        log.info(f"GPU[{i}] {torch.cuda.get_device_name(i)}")


2025-12-10 15:08:42,800 | INFO | CUDA available: True
2025-12-10 15:08:42,826 | INFO | GPU[0] Tesla V100S-PCIE-32GB
2025-12-10 15:08:42,827 | INFO | GPU[1] Tesla V100S-PCIE-32GB


In [9]:
FULL_SHARDS_DIR = Path("data/wiki18/shards_full")
FULL_INDEX_DIR  = Path("index/bm25_full")

RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)

RETR_DIR = RUNS / "retrieval"
ANS_DIR  = RUNS / "answers"
UE_DIR   = RUNS / "ue"
REPORTS_DIR = Path("reports")

for p in [RETR_DIR, ANS_DIR, UE_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

banner("Directories prepared")


2025-12-10 15:08:42,998 | INFO | ================================================================================
2025-12-10 15:08:42,999 | INFO | *** Directories prepared ***
2025-12-10 15:08:42,999 | INFO | ================================================================================


## Download wiki-18 Dataset (idempotent)
Checks if already downloaded in HF cache.  
If not present → downloads only `.jsonl.gz`.


In [10]:
from huggingface_hub import snapshot_download

HF_DATA_REPO = "PeterJinGo/wiki-18-corpus"

def download_wiki18():
    banner("DOWNLOAD OR USE CACHED WIKI-18")
    path = snapshot_download(
        repo_id=HF_DATA_REPO,
        repo_type="dataset",
        allow_patterns=["*.jsonl.gz"],
        local_files_only=False
    )
    snapshot = Path(path)
    candidates = list(snapshot.rglob("*.jsonl.gz"))
    if not candidates:
        raise RuntimeError("No wiki18 .jsonl.gz found")
    return candidates[0]

wiki_gz = download_wiki18()
log.info(f"wiki gz: {wiki_gz}")


2025-12-10 15:09:05,914 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:09:23,200 | INFO | ================================================================================
2025-12-10 15:09:23,200 | INFO | *** DOWNLOAD OR USE CACHED WIKI-18 ***
2025-12-10 15:09:23,201 | INFO | ================================================================================
2025-12-10 15:09:23,401 | INFO | wiki gz: /d/hpc/projects/FRI/ma76193/hf_home/hub/datasets--PeterJinGo--wiki-18-corpus/snapshots/69c1c00ffe7c5554c68d8548355cb22e46aabc51/wiki-18.jsonl.gz


## Stream wiki-18 into shard files  
Only runs if shards do not already exist.


In [11]:
import gzip

def shard_wiki18(gz_path, out_dir, shard_size=100_000):
    banner("SHARDING WIKI-18")
    out_dir.mkdir(parents=True, exist_ok=True)

    existing = list(out_dir.glob("*.jsonl"))
    if existing:
        log.info(f"Shards already exist: {len(existing)} files. Skipping.")
        return

    total = 0
    idx = 0
    written = 0
    out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")

    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except:
                continue

            text = obj.get("text") or obj.get("contents") or ""
            if not text.strip():
                continue

            rec = {
                "id": obj.get("id") or obj.get("page_id") or str(total),
                "contents": text.strip(),
            }
            out_f.write(json.dumps(rec) + "\n")

            total += 1
            written += 1

            if written >= shard_size:
                out_f.close()
                idx += 1
                written = 0
                out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")

    out_f.close()
    log.info(f"Sharding complete. Total docs: {total}")

shard_wiki18(wiki_gz, FULL_SHARDS_DIR)


2025-12-10 15:09:35,915 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:10:05,916 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:10:08,572 | INFO | ================================================================================
2025-12-10 15:10:08,573 | INFO | *** SHARDING WIKI-18 ***
2025-12-10 15:10:08,573 | INFO | ================================================================================
2025-12-10 15:10:08,576 | INFO | Shards already exist: 211 files. Skipping.


## Build BM25 index using Pyserini  
Uses Lucene.  
Skips if index already exists.


In [12]:
def build_index(input_dir, index_dir, threads=8):
    banner("BM25 INDEX BUILD")
    if index_dir.exists() and any(index_dir.iterdir()):
        log.info("Index already exists. Skipping.")
        return

    cmd = [
        sys.executable, "-m", "pyserini.index.lucene",
        "--collection", "JsonCollection",
        "--input", str(input_dir),
        "--index", str(index_dir),
        "--generator", "DefaultLuceneDocumentGenerator",
        "--threads", str(threads),
        "--storePositions",
        "--storeDocvectors",
        "--storeRaw"
    ]
    log.info(" ".join(cmd))
    subprocess.run(cmd, check=True)

build_index(FULL_SHARDS_DIR, FULL_INDEX_DIR)


2025-12-10 15:10:09,122 | INFO | ================================================================================
2025-12-10 15:10:09,122 | INFO | *** BM25 INDEX BUILD ***
2025-12-10 15:10:09,123 | INFO | ================================================================================
2025-12-10 15:10:09,124 | INFO | Index already exists. Skipping.


## Sample Queries  
Given a JSONL file with queries (FactScore-Bio etc.), we sample `n_queries=3`  
This cell is idempotent and overwrites only if missing.


In [13]:
def extract_query_text(obj):
    for k in ["query","question","prompt","instruction","text","claim"]:
        if k in obj and isinstance(obj[k], str) and obj[k].strip():
            return obj[k].strip()
    # fallback
    for v in obj.values():
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None


def sample_queries(src_path: Path, out_path: Path, n: int, seed: int):
    banner("LOAD + SAMPLE QUERIES")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        log.info(f"Already exists → {out_path}")
        return out_path

    items = []
    with open(src_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except:
                continue

            q = extract_query_text(obj)
            if not q:
                continue

            qid = str(obj.get("id") or obj.get("_id") or f"s{i:08d}")
            items.append((qid, q))

    random.Random(seed).shuffle(items)
    picked = items[:n]

    with open(out_path, "w", encoding="utf-8") as f:
        for i, (qid, q) in enumerate(picked, 1):
            sid = f"b{i:03d}"
            f.write(json.dumps({"id": sid, "orig_id": qid, "query": q}, ensure_ascii=False) + "\n")

    log.info(f"Wrote {n} sampled queries → {out_path}")
    return out_path


# YOUR SETTINGS
query_src = Path("data/queries/factscore_bio.jsonl")
sampled_queries = Path("data/queries/notebook.seed1337.jsonl")

sample_queries(query_src, sampled_queries, n=50, seed=1337)


2025-12-10 15:10:09,625 | INFO | ================================================================================
2025-12-10 15:10:09,625 | INFO | *** LOAD + SAMPLE QUERIES ***
2025-12-10 15:10:09,626 | INFO | ================================================================================
2025-12-10 15:10:09,629 | INFO | Already exists → data/queries/notebook.seed1337.jsonl


PosixPath('data/queries/notebook.seed1337.jsonl')

In [14]:
!module avail 2>&1 | grep -i java


   ANTLR/2.7.7-GCCcore-10.3.0-Java-11
   ANTLR/2.7.7-GCCcore-11.3.0-Java-11                      (D)
   Java/1.8.0_162
   Java/1.8.0_202                                          (1.8)
   Java/11.0.2                                             (D:11)
   RDP-Classifier/2.13-Java-11
   Trimmomatic/0.39-Java-11
   ant/1.10.11-Java-11


In [15]:
!java -version
!find $JAVA_HOME -name "libjvm.so"


Picked up JAVA_TOOL_OPTIONS: -Xms1g -Xmx8g
openjdk version "21.0.1" 2023-10-17 LTS
OpenJDK Runtime Environment Temurin-21.0.1+12 (build 21.0.1+12-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.1+12 (build 21.0.1+12-LTS, mixed mode, sharing)
/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so


## Retrieval + Reranking  
1. BM25 retrieves top-Kfirst (1000)  
2. CrossEncoder reranks  
3. Keep top-Kkeep (3)  
Everything saved to JSONL.


In [ ]:
from pyserini.search.lucene import LuceneSearcher
from sentence_transformers import CrossEncoder

def get_doc_text(raw):
    try:
        obj = json.loads(raw)
        return obj.get("contents", "").strip()
    except:
        return str(raw).strip()


def retrieve_rerank(queries_path: Path,
                    index_dir: Path,
                    out_path: Path,
                    k_first=1000,
                    k_keep=3,
                    ce_model="cross-encoder/ms-marco-MiniLM-L6-v2",
                    batch_size=64):

    banner("BM25 → CE RERANK")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Resume
    done = {}
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    ex = json.loads(line)
                    done[ex["id"]] = ex
                except:
                    pass
        log.info(f"Resume: {len(done)} rows")

    # CrossEncoder
    try:
        ce = CrossEncoder(ce_model, device="cuda")
    except Exception as e:
        log.warning(f"CE unavailable: {e}")
        ce = None

    searcher = LuceneSearcher(str(index_dir))
    searcher.set_bm25(k1=0.9, b=0.4)

    new = 0
    with open(queries_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:

        for line in fin:
            ex = json.loads(line)
            qid, query = ex["id"], ex["query"]

            if qid in done:
                continue

            hits = searcher.search(query, k_first)
            if not hits:
                fout.write(json.dumps({"id": qid, "query": query, "docs": []}) + "\n")
                new += 1
                continue

            candidates = []
            for h in hits:
                raw = searcher.doc(h.docid).raw()
                text = get_doc_text(raw)
                candidates.append({
                    "docid": h.docid,
                    "raw": raw,
                    "bm25": float(h.score),
                    "text": text
                })

            # Rerank
            if ce:
                pairs = [(query, c["text"]) for c in candidates]
                try:
                    scores = ce.predict(pairs, batch_size=batch_size)
                    for c, s in zip(candidates, scores):
                        c["ce"] = float(s)
                    candidates.sort(key=lambda x: x.get("ce", -1e9), reverse=True)
                except Exception as e:
                    log.warning(f"CE failed: {e}")
                    candidates.sort(key=lambda x: x["bm25"], reverse=True)
            else:
                candidates.sort(key=lambda x: x["bm25"], reverse=True)

            final_docs = [
                {"docid": c["docid"], "raw": c["raw"], "bm25": c["bm25"], "ce": c.get("ce")}
                for c in candidates[:k_keep]
            ]

            fout.write(json.dumps({"id": qid, "query": query, "docs": final_docs}) + "\n")
            new += 1
            log.info(f"[{qid}] top docs: {len(final_docs)}")

    log.info(f"Rerank complete — new: {new}")
    return out_path


retr_file = RETR_DIR / "notebook.seed1337.rerank3.jsonl"
retrieve_rerank(sampled_queries, FULL_INDEX_DIR, retr_file, k_first=1000, k_keep=3)


2025-12-10 15:10:54,512 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:11:24,513 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:11:54,515 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:12:24,516 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:12:54,517 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:13:24,518 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:13:30,127 | INFO | 
Using override env var JVM_PATH (/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12/lib/server/libjvm.so) to load libjvm.
Please report your system information (os version, java
version, etc), and the path that works for you, to the
PyJNIus project, at https://github.com/kivy/pyjnius/issues.
so we can improve the automatic discovery.



Picked up JAVA_TOOL_OPTIONS: -Xms1g -Xmx8g


2025-12-10 15:13:54,519 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:13:56,613 | INFO | Loading faiss with AVX2 support.
2025-12-10 15:14:27,002 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:14:27,011 | INFO | Successfully loaded faiss with AVX2 support.
2025-12-10 15:14:57,012 | INFO | [HEARTBEAT] Notebook run alive...


/d/hpc/home/ma76193/.local/lib/python3.11/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


2025-12-10 15:15:27,013 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:15:57,014 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:16:27,015 | INFO | [HEARTBEAT] Notebook run alive...
2025-12-10 15:16:57,017 | INFO | [HEARTBEAT] Notebook run alive...


In [ ]:
import sys
print(sys.executable)
print(sys.version)

import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))


## Load Qwen2.5-7B-Instruct (bf16)  
Loads from HF cache if available.  
Used for answer generation with strict "docs-only" policy.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def parse_docs_for_prompt(docs):
    texts = []
    for i, d in enumerate(docs, 1):
        try:
            obj = json.loads(d["raw"])
            text = obj.get("contents", "")
        except:
            text = d.get("raw", "")
        text = " ".join(text.split())
        if len(text) > 1200:
            text = text[:1195] + " ..."
        texts.append(f"[d{i}] {text}")
    return "\n".join(texts)


def build_prompt(query, docs, strict=True):
    policy = (
        "Answer only using the provided documents. If the documents do not contain the answer, say you don't know."
        if strict else
        "Prefer the provided documents; if insufficient, you may use general knowledge."
    )
    return f"You are a careful assistant. {policy}\n\nQuestion: {query}\n\nDocuments:\n{parse_docs_for_prompt(docs)}\n\nAnswer:"


def load_qwen(model_id="Qwen/Qwen2.5-7B-Instruct"):
    banner("LOAD QWEN 7B (bf16)")
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if torch.cuda.is_available():
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        device = "cuda"
    else:
        mdl = AutoModelForCausalLM.from_pretrained(model_id)
        device = "cpu"

    log.info(f"Loaded Qwen on {device}")
    return tok, mdl, device


qwen_tok, qwen_mdl, qwen_device = load_qwen()


## Generate Answers (Strict Docs-Only)
Uses Qwen2.5-7B-Instruct (bf16).  
Notebook version supports resume: existing IDs in output JSONL are skipped.


In [ ]:
from transformers import TextStreamer

def generate_answers(retr_path: Path,
                     out_path: Path,
                     tok,
                     mdl,
                     device,
                     max_new_tokens=256,
                     strict=True):

    banner("GENERATE ANSWERS — Qwen 7B (bf16)")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done = set()
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except:
                    pass
        log.info(f"Resuming — already have {len(done)} answers.")

    new = 0

    with open(retr_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:

        for line in fin:
            ex = json.loads(line)
            qid = ex["id"]
            q = ex["query"]
            docs = ex["docs"]

            if qid in done:
                continue

            prompt = build_prompt(q, docs, strict=strict)

            enc = tok(prompt, return_tensors="pt")
            if device == "cuda":
                enc = {k: v.cuda() for k, v in enc.items()}

            streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)

            out = mdl.generate(
                **enc,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
                streamer=streamer
            )

            full_text = tok.decode(out[0], skip_special_tokens=True)
            # Extract answer
            if "Answer:" in full_text:
                answer = full_text.split("Answer:", 1)[-1].strip()
            else:
                answer = full_text.strip()

            record = {
                "id": qid,
                "query": q,
                "answer": answer,
                "docs": docs,
                "meta": {
                    "ts": datetime.now().isoformat(timespec="seconds"),
                    "model": "Qwen/Qwen2.5-7B-Instruct",
                    "dtype": "bf16" if device == "cuda" else "fp32",
                    "max_new_tokens": max_new_tokens
                }
            }

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            new += 1

            log.info(f"[{qid}] answer length = {len(answer)}")

    log.info(f"Answer generation complete — new answers: {new}")
    return out_path


answers_file = ANS_DIR / "notebook.seed1337.qwen7b.jsonl"
generate_answers(retr_file, answers_file, qwen_tok, qwen_mdl, qwen_device)


## SAFE-Lite (Classifier-Only)
Uses `safe-ai/SAFE-Classifier`  
Generates `safe_score` between 0–1  
Higher = more factual confidence.


In [ ]:
from pathlib import Path

# This is the file containing your Qwen responses to be evaluated
ans_file = Path("/d/hpc/projects/FRI/ma76193/IR_Project/src/runs/answers/notebook.seed1337.qwen7b.jsonl")

assert ans_file.exists(), f"Answer file not found: {ans_file}"
print("[SAFE] Evaluating responses from:", ans_file)


In [ ]:
import sys
for k in list(sys.modules.keys()):
    if "TruthTorchLM.long_form_generation.utils" in k:
        del sys.modules[k]
print("Utils cache cleared.")


In [ ]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)


In [ ]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print("Functions:", [f for f in dir(su) if "extract" in f or "strip" in f])


In [ ]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)


In [ ]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(su.__file__)
print("extract_first_line exists:", hasattr(su, "extract_first_line"))


In [ ]:
import TruthTorchLM.long_form_generation.utils.safe_utils as su
print(dir(su))


In [ ]:
# === SAFE POST-RAG FACT CHECKING WITH QWEN ===========================
print("=== INIT SAFE (QWEN) ===")

import importlib.util
from pathlib import Path
import json
import pandas as pd

# Path to your LOCAL eval_claim.py (already modified to include reranker)
LOCAL_EVAL_CLAIM = Path(
    "/d/hpc/projects/FRI/ma76193/IR_Project/src/TruthTorchLM/src/TruthTorchLM/long_form_generation/evaluators/eval_claim.py"
)

spec = importlib.util.spec_from_file_location("safe_eval_claim", LOCAL_EVAL_CLAIM)
safe_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(safe_module)
ClaimEvaluator = safe_module.ClaimEvaluator

print(f"[SAFE] Loaded ClaimEvaluator from {LOCAL_EVAL_CLAIM}")

# Use already-loaded Qwen model and tokenizer
safe_tok = qwen_tok
safe_mdl = qwen_mdl

print("[SAFE] Using existing Qwen2.5-7B-Instruct model already in memory.")

safe_eval = ClaimEvaluator(
    rater_model=safe_mdl,
    rater_tokenizer=safe_tok,
    lucene_index_dir=str(FULL_INDEX_DIR),
    max_steps=3,
    max_retries=3,
    bm25_k=3,
)

safe_jsonl = UE_DIR / "notebook.seed1337.safe.jsonl"
safe_csv = UE_DIR / "notebook.seed1337.safe.csv"

rows = []

print("=== RUN SAFE ===")

with open(ans_file, "r") as fin, open(safe_jsonl, "w") as fout:
    for line in fin:
        ex = json.loads(line)
        claim = ex["answer"]

        print(f"\n[SAFE] Processing ID={ex['id']}")
        res = safe_eval(claim)

        row = {
            "id": ex["id"],
            "safe_score": res["answer"],
            "safe_response": res["response"],
            "safe_details": res["search_details"],
        }

        rows.append(row)
        fout.write(json.dumps(row) + "\n")

# Save CSV
pd.DataFrame(rows).to_csv(safe_csv, index=False)

print("SAFE completed →", safe_jsonl)
print("SAFE CSV saved →", safe_csv)


## MARS (White-box UE)
Notebook version:
- Tries TruthTorchLM (if installed)
- Falls back to NLL(answer|prompt)
All logs printed + saved.


In [ ]:
from pathlib import Path
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_mars(answers_file: Path, out_jsonl: Path, out_csv: Path, model_id="Qwen/Qwen2.5-7B-Instruct"):
    print("=== MARS WHITE-BOX ===")

    # Try TruthTorchLM's implementation
    use_tt = False
    try:
        from truthtorchlm import mars as tt_mars
        use_tt = True
        print("Using TruthTorchLM MARS.")
    except Exception:
        print("TruthTorchLM unavailable → fallback.")
        use_tt = False

    # Prepare fallback LM
    tok, mdl = None, None
    if not use_tt:
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
        )

    rows = []

    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)
            qid = ex["id"]
            q = ex["query"]
            ans = ex["answer"]
            docs = ex["docs"]

            score = None
            details = {}

            # 1. TruthTorchLM MARS
            if use_tt:
                try:
                    score, details = tt_mars.compute(
                        question=q,
                        answer=ans,
                        docs=[d.get("raw","") for d in docs],
                        model_name=model_id
                    )
                except Exception as e:
                    print(f"[{qid}] TT error: {e}")

            # 2. Fallback MARS (NLL-based)
            if score is None:
                prompt = build_prompt(q, docs, strict=True)
                full = prompt + "\n\nAnswer: " + ans

                enc = tok(full, return_tensors="pt")
                if torch.cuda.is_available():
                    enc = {k: v.cuda() for k, v in enc.items()}

                # Mask instructions
                prompt_ids = tok(prompt + "\n\nAnswer:", return_tensors="pt")["input_ids"][0]
                full_ids = enc["input_ids"][0]

                labels = full_ids.clone()
                labels[: prompt_ids.size(0)] = -100

                out = mdl(**enc, labels=labels.unsqueeze(0))
                nll = float(out.loss.detach().cpu())
                score = -nll
                details = {"fallback": "nll"}

            rows.append({
                "id": qid,
                "mars_score": score,
                "mars_details": details
            })

            print(f"[{qid}] MARS = {score}")

    # Save JSONL
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    # Save CSV
    pd.DataFrame(rows).to_csv(out_csv, index=False)

    print("MARS done →", out_jsonl)
    print("MARS CSV saved →", out_csv)

    return out_jsonl, out_csv


mars_jsonl = UE_DIR / "notebook.seed1337.mars.jsonl"
mars_csv   = UE_DIR / "notebook.seed1337.mars.csv"

run_mars(answers_file, mars_jsonl, mars_csv)


## Eccentricity (Black-box UE)
Embedding-based OOD check.  
Compute:  
- Answer embedding  
- Context centroid  
- Cosine distance  
- Z-score normalization


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json, csv
from pathlib import Path

def run_ecc(answers_file: Path, jsonl_out: Path, csv_out: Path,
            embed_model="sentence-transformers/all-MiniLM-L6-v2"):

    banner("ECCENTRICITY")

    model = SentenceTransformer(embed_model, device="cuda")

    rows = []

    with open(answers_file, "r") as f:
        for line in f:
            ex = json.loads(line)

            qid = ex["id"]
            ans = ex["answer"]
            docs = ex["docs"]

            ctx = [get_doc_text(d.get("raw",""))[:1000] for d in docs if get_doc_text(d.get("raw",""))]

            if not ctx:
                rows.append({"id": qid, "ecc": None, "ecc_z": None})
                continue

            v_ans = model.encode([ans], normalize_embeddings=True)[0]
            v_ctx = model.encode(ctx, normalize_embeddings=True)

            centroid = v_ctx.mean(0)
            cos_sim = float((v_ans * centroid).sum())
            ecc = 1 - cos_sim

            rows.append({"id": qid, "ecc": ecc, "ecc_z": None})

    # compute z-score
    vals = np.array([r["ecc"] for r in rows if r["ecc"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), vals.std(ddof=1) or 1.0
        for r in rows:
            if r["ecc"] is not None:
                r["ecc_z"] = float((r["ecc"] - mu) / sd)

    # save jsonl
    with open(jsonl_out, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

    # save csv
    with open(csv_out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "ecc", "ecc_z"])
        w.writeheader()
        for r in rows:
            w.writerow(r)

    return jsonl_out, csv_out


run_ecc(
    answers_file,
    UE_DIR / "factcheck.ecc.jsonl",
    UE_DIR / "factcheck.ecc.csv"
)


In [ ]:
from pathlib import Path

# BASE PROJECT ROOT
ROOT = Path("/d/hpc/projects/FRI/ma76193/IR_Project/src")

# Correct folders
QUERIES_DIR = ROOT / "data" / "queries"
ANSWERS_DIR = ROOT / "runs" / "answers"
UE_DIR      = ROOT / "runs" / "ue"

# Correct file names (from your folders)
sampled_queries = QUERIES_DIR / "notebook.seed1337.jsonl"
answers_file    = ANSWERS_DIR / "notebook.seed1337.qwen7b.jsonl"

mars_file = UE_DIR / "notebook.seed1337.mars.jsonl"
ecc_file  = UE_DIR / "notebook.seed1337.ecc.jsonl"
safe_file = UE_DIR / "notebook.seed1337.safe.jsonl"

# Output reports into runs/
REPORTS_DIR = ROOT / "runs"
final_md  = REPORTS_DIR / "notebook.seed1337.report.md"
final_csv = REPORTS_DIR / "notebook.seed1337.scores.csv"
corr_csv  = REPORTS_DIR / "notebook.seed1337.correlations.csv"

print("Queries file: ", sampled_queries.exists(), sampled_queries)
print("Answers file: ", answers_file.exists(), answers_file)
print("MARS file: ", mars_file.exists(), mars_file)
print("ECC file: ", ecc_file.exists(), ecc_file)
print("SAFE file:", safe_file.exists(), safe_file)


## Write Combined CSV + Markdown Report


In [ ]:
import json, csv
import pandas as pd
import numpy as np
from pathlib import Path
# Re-declare evaluation output files exactly as your SAFE/MARS/ECC cells produced them


def write_final_report(
    queries_file: Path,
    answers_file: Path,
    mars_file: Path,
    ecc_file: Path,
    safe_file: Path,
    out_md: Path,
    out_csv: Path,
    corr_csv: Path
):
    banner("FINAL REPORT")

    # -----------------------------
    # Load data dictionaries
    # -----------------------------
    qs = {json.loads(line)["id"]: json.loads(line)
          for line in open(queries_file, "r")}

    mars = {json.loads(line)["id"]: json.loads(line)
            for line in open(mars_file, "r")}

    ecc = {json.loads(line)["id"]: json.loads(line)
           for line in open(ecc_file, "r")}

    safe = {json.loads(line)["id"]: json.loads(line)
            for line in open(safe_file, "r")}

    # -----------------------------
    # Build master table in memory
    # -----------------------------
    rows = []

    for line in open(answers_file, "r"):
        ex = json.loads(line)
        qid = ex["id"]

        safe_label = safe.get(qid, {}).get("safe_score")

        # correctness proxy (Supported=1, Not Supported=0)
        if safe_label == "Supported":
            correctness = 1
        elif safe_label == "Not Supported":
            correctness = 0
        else:
            correctness = None

        rows.append({
            "id": qid,
            "query": ex["query"],
            "answer": ex["answer"],
            "mars": mars.get(qid, {}).get("mars_score"),
            "ecc": ecc.get(qid, {}).get("ecc"),
            "ecc_z": ecc.get(qid, {}).get("ecc_z"),
            "safe": safe_label,
            "correctness": correctness,
        })

    df = pd.DataFrame(rows)

    # Save main CSV
    df.to_csv(out_csv, index=False)

    # -----------------------------
    # Save Markdown summary
    # -----------------------------
    with open(out_md, "w") as f:
        f.write("# Notebook RAG Evaluation Report\n\n")
        f.write("| ID | Query | SAFE | MARS | ECC_z | Correctness |\n")
        f.write("|----|-------|------|------|--------|-------------|\n")

        for _, r in df.iterrows():
            q_short = r["query"][:120].replace("|", "/")
            a_safe = r["safe"]
            f.write(f"| {r['id']} | {q_short} | {a_safe} | {r['mars']} | {r['ecc_z']} | {r['correctness']} |\n")

    # -----------------------------
    # CORRELATION ANALYSIS
    # -----------------------------
    corr_df = df[["correctness", "mars", "ecc", "ecc_z"]].dropna()

    correlations = corr_df.corr(method="pearson")
    correlations.to_csv(corr_csv)

    print("\n=== CORRELATION MATRIX (Correctness vs Uncertainty) ===")
    print(correlations)

    banner("DONE")


# -----------------------------
# Paths (same names you used)
# -----------------------------
final_md  = REPORTS_DIR / "notebook.seed1337.report.md"
final_csv = REPORTS_DIR / "notebook.seed1337.scores.csv"
corr_csv  = REPORTS_DIR / "notebook.seed1337.correlations.csv"

write_final_report(
    sampled_queries,
    answers_file,
    mars_file,
    ecc_file,
    safe_json,     # your safe scores file
    final_md,
    final_csv,
    corr_csv
)
